# An SBML cell-cycle model in every cell — analysis

This notebook accompanies `sbml_cell_cycle.py` and demonstrates two things:

1. **SBML integration with tyssue.**
   - A BioModels SBML file is read directly and compiled once into a vectorised
     numpy right-hand side (`vivarium_tyssue.sbml.VectorizedSBML`).
   - There is no COPASI or roadrunner call per cell. All cells are integrated
     together as one `(n_species, n_cells)` array.
2. **A subcellular model running in every cell.**
   - The `SBMLCellCycle` process gives every face of the vertex model its own copy
     of the model.
   - Each cell reads its *own* cell-cycle phase from its cyclins: it grows in G1
     and G2, holds its size in S and M, and divides on entering M.
   - Daughters inherit their mother's molecular state.
   - The tissue has a free border, so it spreads as it grows.

The notebook has four parts:
1. **The model:** its equations and parameters, and a check that the compiled SBML
   is exactly these equations.
2. **Single-cell behaviour:** the oscillation, the growth-factor switch, and the
   commitment point.
3. **The coupling to the vertex model.**
4. **The growing tissue** read back from `outputs/`, with GIFs.

Run `python sbml_cell_cycle.py` first.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import PolyCollection
from IPython.display import display, Markdown

import sbml_cell_cycle as sim          # the run script is the single source of truth
sys.path.insert(0, str(sim.REPO))
from vivarium_tyssue.sbml import VectorizedSBML

FIG_DPI = 300   # saved figures; inline display stays at 150
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": FIG_DPI, "axes.spines.top": False, "axes.spines.right": False})
FIG_DIR = sim.OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

model = VectorizedSBML(sim.MODEL_SBML)
SHORT = {"cyclin_D_Cdk4_6": "Md", "transcription_factor_E2F_active": "E2F",
         "cyclin_E_Cdk2": "Me", "cyclin_A_Cdk2": "Ma", "cyclin_B_Cdk1": "Mb",
         "Cdc20_active": "Cdc20"}
COLORS = {"Md": "#1b9e77", "E2F": "#999999", "Me": "#d95f02", "Ma": "#7570b3",
          "Mb": "#e7298a", "Cdc20": "#66a61e"}
IB = model.index("cyclin_B_Cdk1")
print(model.model_name)
print("state:", model.state_ids)

## 1. The model — Gérard & Goldbeter 2011 skeleton

**Source.**
- C. Gérard & A. Goldbeter, *A skeleton model for the network of cyclin-dependent
  kinases driving the mammalian cell cycle*, Interface Focus **1**, 24–35 (2011).
- BioModels [BIOMD0000000941](https://www.ebi.ac.uk/biomodels/BIOMD0000000941),
  curated, with a figure of the paper reproduced by the curators.

**What kind of model it is.**
- It describes **one cell**: every variable is an intracellular concentration, and
  there are no population variables.
- It is the backbone of the authors' 39-variable model of the mammalian Cdk network
  (BIOMD0000000730).
- It keeps only the four cyclin/Cdk modules and the feedback loops that make the
  cycle oscillate and switch. It has no Cdk inhibitors, no pRB and no cell mass.

| symbol | SBML species | meaning |
|---|---|---|
| $M_d$ | `cyclin_D_Cdk4_6` | cyclin D/Cdk4–6, which drives G1 progression |
| $E2F$ | `transcription_factor_E2F_active` | active E2F transcription factor |
| $M_e$ | `cyclin_E_Cdk2` | cyclin E/Cdk2, the G1/S transition |
| $M_a$ | `cyclin_A_Cdk2` | cyclin A/Cdk2, progression through S and G2 |
| $M_b$ | `cyclin_B_Cdk1` | cyclin B/Cdk1, the G2/M transition (mitosis) |
| $Cdc20$ | `Cdc20_active` | active Cdc20 (APC/C), which drives mitotic exit |

### Equations

$$
\begin{aligned}
\frac{dM_d}{dt} &= v_{sd}\,\frac{GF}{K_{gf}+GF} \;-\; V_{dd}\,\frac{M_d}{K_{dd}+M_d}\\[4pt]
\frac{dE2F}{dt} &= V_{1e2f}\,\frac{E2F_{tot}-E2F}{K_{1e2f}+E2F_{tot}-E2F}\,\big(M_d+M_e\big) \;-\; V_{2e2f}\,\frac{E2F}{K_{2e2f}+E2F}\,M_a\\[4pt]
\frac{dM_e}{dt} &= v_{se}\,E2F \;-\; V_{de}\,M_a\,\frac{M_e}{K_{de}+M_e}\\[4pt]
\frac{dM_a}{dt} &= v_{sa}\,E2F \;-\; V_{da}\,Cdc20\,\frac{M_a}{K_{da}+M_a}\\[4pt]
\frac{dM_b}{dt} &= v_{sb}\,M_a \;-\; V_{db}\,Cdc20\,\frac{M_b}{K_{db}+M_b}\\[4pt]
\frac{dCdc20}{dt} &= V_{1cdc20}\,M_b\,\frac{Cdc20_{tot}-Cdc20}{K_{1cdc20}+Cdc20_{tot}-Cdc20} \;-\; V_{2cdc20}\,\frac{Cdc20}{K_{2cdc20}+Cdc20}
\end{aligned}
$$

$E2F_{tot}$ and $Cdc20_{tot}$ are conserved totals. In the SBML they are species
that no reaction changes, so the engine treats them as constants. The only input
from outside the cell is the growth factor $GF$.

### The feedback logic

1. **Growth factor → cyclin D.** Cyclin D is synthesised in proportion to
   $GF/(K_{gf}+GF)$ and degraded with Michaelis–Menten kinetics.
2. **The E2F switch.** Cyclin D, and then cyclin E, activate E2F. The activation
   and inactivation terms are both zero-order ultrasensitive ($K_{1e2f}$ and
   $K_{2e2f}$ are 0.01), so E2F flips on and off like a switch.
3. **Positive feedback at G1/S.** E2F makes cyclin E, and cyclin E activates E2F.
   Once the loop starts it runs on its own, which commits the cell to S phase.
4. **Negative feedback in S and G2.** E2F also makes cyclin A. Cyclin A switches E2F
   off and drives cyclin E degradation, which ends G1/S.
5. **G2/M.** Cyclin A drives cyclin B synthesis, and cyclin B peaks at mitosis.
6. **Mitotic exit.** Cyclin B activates Cdc20. Cdc20 destroys cyclins A and B,
   cyclin B collapses, and the cell leaves mitosis. With cyclin A gone, E2F can come
   back on and the loop restarts.

In [ ]:
MEANING = {
    "GF": "growth factor (the external input)",
    "vsd": "max synthesis rate of cyclin D", "Kgf": "GF half-saturation",
    "Vdd": "max degradation rate of cyclin D", "Kdd": "Michaelis constant, cyclin D degradation",
    "V1e2f": "E2F activation rate (by cyclins D and E)", "K1e2f": "Michaelis constant, E2F activation",
    "V2e2f": "E2F inactivation rate (by cyclin A)", "K2e2f": "Michaelis constant, E2F inactivation",
    "E2F_total": "total E2F (conserved)",
    "vse": "cyclin E synthesis rate (per unit E2F)", "Vde": "cyclin E degradation rate (by cyclin A)",
    "Kde": "Michaelis constant, cyclin E degradation",
    "vsa": "cyclin A synthesis rate (per unit E2F)", "Vda": "cyclin A degradation rate (by Cdc20)",
    "Kda": "Michaelis constant, cyclin A degradation",
    "vsb": "cyclin B synthesis rate (per unit cyclin A)", "Vdb": "cyclin B degradation rate (by Cdc20)",
    "Kdb": "Michaelis constant, cyclin B degradation",
    "V1cdc20": "Cdc20 activation rate (by cyclin B)", "K1cdc20": "Michaelis constant, Cdc20 activation",
    "V2cdc20": "Cdc20 inactivation rate", "K2cdc20": "Michaelis constant, Cdc20 inactivation",
    "Cdc20_total": "total Cdc20 (conserved)", "nuclear": "compartment volume",
}
params = model.default_params()
table = pd.DataFrame(
    [(k, params[k], MEANING.get(k, "")) for k in MEANING if k in params],
    columns=["parameter", "SBML value", "meaning"],
)
initial = pd.DataFrame({"species": model.state_ids, "symbol": [SHORT[s] for s in model.state_ids],
                        "initial value": model.initial_state()})
display(table.style.hide(axis="index").format({"SBML value": "{:g}"}))
display(initial.style.hide(axis="index"))

### From SBML to numpy, and a check that it is the model above

`VectorizedSBML` performs these steps once:
- expand the SBML function definitions;
- translate every kinetic law's MathML into Python;
- divide each rate by the compartment volume (species are concentrations);
- compile the result into one function.

The generated source is printed below. `Y[i]` is row *i* of the
`(n_species, n_cells)` state, and each `P[...]` is a scalar or a per-cell array.

The check below this evaluates the equations written out above, by hand, on 1000
random states. It confirms they agree with the compiled SBML to machine precision.
So the LaTeX above is exactly what the simulation integrates.

In [ ]:
print(model.source)

In [ ]:
def rhs_from_equations(y, p):
    Md, E2F, Me, Ma, Mb, C = y
    return np.array([
        p["vsd"] * p["GF"] / (p["Kgf"] + p["GF"]) - p["Vdd"] * Md / (p["Kdd"] + Md),
        p["V1e2f"] * (p["E2F_total"] - E2F) / (p["K1e2f"] + p["E2F_total"] - E2F) * (Md + Me)
        - p["V2e2f"] * E2F / (p["K2e2f"] + E2F) * Ma,
        p["vse"] * E2F - p["Vde"] * Ma * Me / (p["Kde"] + Me),
        p["vsa"] * E2F - p["Vda"] * C * Ma / (p["Kda"] + Ma),
        p["vsb"] * Ma - p["Vdb"] * C * Mb / (p["Kdb"] + Mb),
        p["V1cdc20"] * Mb * (p["Cdc20_total"] - C) / (p["K1cdc20"] + p["Cdc20_total"] - C)
        - p["V2cdc20"] * C / (p["K2cdc20"] + C),
    ])

rng = np.random.default_rng(0)
Y = rng.uniform(0, 2.9, size=(6, 1000))
P = model.params({"GF": rng.uniform(0, 2, 1000)})     # a different GF in every "cell"
err = np.max(np.abs(model.rhs(Y, P) - rhs_from_equations(Y, P)) / (1 + np.abs(rhs_from_equations(Y, P))))
print(f"max relative difference, compiled SBML vs equations: {err:.2e}")
assert err < 1e-12

## 2. Single-cell behaviour

The integrator is the same one the tissue simulation uses: clipped explicit Euler
with a step of 0.01 h of model time, within 0.2% of LSODA on the period. Clipping
matters because cyclin B falls to about 1e-5 at mitotic exit, where an unclipped
step can overshoot below zero.

### 2a. The oscillation at GF = 1

The cyclins peak one after another, $M_d \to E2F \to M_e \to M_a \to M_b \to Cdc20$,
which gives the ordered sequence G1 → S → G2 → M.

The shading shows the phases the tissue reads from these cyclins (Section 3):
cells grow in **G1** and **G2**, hold their size in **S** and **M**, and divide on
entering M.

In [ ]:
y = model.integrate(model.initial_state()[:, None], model.params(), 300.0)[:, 0]
t, S = model.trajectory(y, model.params(), 60.0)
fig, ax = plt.subplots(figsize=(8, 3.4))
for i, sid in enumerate(model.state_ids):
    ax.plot(t, S[:, i], color=COLORS[SHORT[sid]], label=SHORT[sid], lw=1.6)
from vivarium_tyssue.core import build_core
from vivarium_tyssue.processes.cell_cycle import SBMLCellCycle
core = build_core()
cycle_proc = SBMLCellCycle(sim.cell_cycle_config(), core=core)   # the tissue's phase rules
PHASE_COLORS = {"G1": "#fde0c5", "S": "#9ecae1", "G2": "#3182bd", "M": "#08306b"}
ph = cycle_proc.classify(S.T)
edges = np.flatnonzero(np.diff(ph)) + 1
for a, b in zip(np.r_[0, edges], np.r_[edges, len(ph) - 1]):
    ax.axvspan(t[a], t[b], color=PHASE_COLORS[cycle_proc.phase_names[ph[a]]], alpha=0.35, lw=0)
ax.set(xlabel="time (h)", ylabel="concentration (µM)", title="Gérard–Goldbeter skeleton model, GF = 1")
ax.legend(ncol=6, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.22))
fig.tight_layout(); fig.savefig(FIG_DIR / "single_cell_timecourse.png"); plt.show()

ups = t[np.flatnonzero((S[:-1, IB] < 1) & (S[1:, IB] >= 1))]
print(f"period at GF = 1: {np.diff(ups).mean():.2f} h")

### 2b. Growth factor is an on/off switch

Below we run 160 cells at once, each with its own GF, which is exactly how the
tissue integrates its cells.
- Below **GF ≈ 0.072** there is no full cycle. The cell settles into a G0 steady
  state, or at most makes small-amplitude oscillations.
- Above the threshold there are full-amplitude cycles, and **the period barely
  changes with GF**.

The practical consequence: GF (or anything mapped onto it) decides **whether** a
cell cycles, not **how fast**.

In [ ]:
gf = np.concatenate([np.linspace(0.0, 0.2, 121), np.linspace(0.21, 2.0, 39)])
Pg = model.params({"GF": gf})
Y = np.tile(model.initial_state()[:, None], (1, gf.size))
Y = model.integrate(Y, Pg, 1500.0)
h, T = 0.01, 200.0
mb_max, mb_min = np.zeros(gf.size), np.full(gf.size, np.inf)
crossings = [[] for _ in gf]
prev = Y[IB].copy()
for k in range(int(T / h)):
    Y = model.integrate(Y, Pg, h, h)
    mb_max, mb_min = np.maximum(mb_max, Y[IB]), np.minimum(mb_min, Y[IB])
    for i in np.flatnonzero((prev < 1) & (Y[IB] >= 1)):
        crossings[i].append(k * h)
    prev = Y[IB].copy()
period = np.array([np.diff(c).mean() if len(c) > 2 else np.nan for c in crossings])
threshold = gf[np.argmax(mb_max > 1.0)]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))
a1.fill_between(gf, mb_min, mb_max, color=COLORS["Mb"], alpha=0.35, lw=0)
a1.plot(gf, mb_max, color=COLORS["Mb"])
a1.axvline(threshold, color="k", ls="--", lw=0.8)
a1.set(xscale="symlog", xlabel="GF", ylabel="cyclin B range (µM)", title="amplitude")
a2.plot(gf, period, "o", ms=2.5, color="k")
a2.axvline(threshold, color="k", ls="--", lw=0.8)
a2.set(xscale="symlog", xlabel="GF", ylabel="period (h)", title="period")
fig.tight_layout(); fig.savefig(FIG_DIR / "gf_switch.png"); plt.show()
print(f"full cycles for GF >= {threshold:.3f}; period {np.nanmax(period):.1f} -> {np.nanmin(period):.1f}")

### 2c. Where the cell commits: growth factor withdrawal

Starting from a cell cycling at GF = 1, GF is set to 0 at different points in the
cycle. Each row below is one cell, and the | marks when GF is withdrawn.
- **Withdrawn more than ~3 h before mitotic exit:** the cell finishes
  that mitosis and stays in G0.
- **Withdrawn later than that, including any time after exit:** the cell completes
  one more full cycle.

In this skeleton model the decision about the *next* cycle is made **just before
the previous mitotic exit**. At exit cyclin A is gone, and E2F switches on with the
cyclin D and E that remain.

The table below shows the other side of the same window. With GF absent
throughout, **a GF pulse lasting just 2 h** anywhere in the first
~10 h after exit restarts a full cycle.

What this means if GF is later made to depend on the tissue, through
`input_columns`:
- **Overshoot.** Suppose a cell loses GF just after its decision point.
  - It still divides at the end of its current cycle.
  - Both daughters inherit the committed state, so they divide again at the end of
    the next cycle.
  - So up to **4×** as many cells as when GF was lost. Arrest lags any mechanical
    signal by one to two division rounds.
- **The input must not spike right after mitotic exit.** For example, raw cell area
  is a bad input: a cell growing in G1 would give itself exactly such a pulse.
  Strain (`"area/prefered_area"`) avoids this.

In [ ]:
y = model.integrate(model.initial_state()[:, None], model.params(), 300.0)[:, 0]
t, S = model.trajectory(y, model.params(), 60.0)
exit_i = np.flatnonzero((S[:-1, IB] >= 0.1) & (S[1:, IB] < 0.1))[0]
y_exit, t_exit = S[exit_i], t[exit_i]
offsets = np.arange(-6.0, 17.0, 2.0)               # withdrawal time relative to exit
start = model.integrate(y[:, None], model.params(), t_exit - 6.0)   # 6 h before exit
Y = np.tile(start, (1, offsets.size))
h, T = 0.01, 80.0
rec = np.empty((int(T / h), offsets.size))
for k in range(rec.shape[0]):
    tk = -6.0 + k * h
    Y = model.integrate(Y, model.params({"GF": np.where(tk < offsets, 1.0, 0.0)}), h, h)
    rec[k] = Y[IB]
tt = -6.0 + h * np.arange(rec.shape[0])

fig, ax = plt.subplots(figsize=(8, 4))
for j, off in enumerate(offsets):
    ax.plot(tt, rec[:, j] / 3.2 + j, color=COLORS["Mb"], lw=1.2)
    ax.plot([off], [j], "k|", ms=10)
    extra = int(((rec[:-1, j] < 1) & (rec[1:, j] >= 1)).sum())
    ax.text(T - 6.0 + 1, j + 0.1, f"{extra} more", fontsize=7, va="bottom")
ax.axvline(0, color="k", ls=":", lw=0.8)
ax.set_yticks(range(offsets.size), [f"{o:+.0f}" for o in offsets])
ax.set(xlabel="time relative to mitotic exit (h)", ylabel="GF withdrawn at (| mark)",
       title="cyclin B after growth-factor withdrawal")
fig.tight_layout(); fig.savefig(FIG_DIR / "gf_withdrawal.png"); plt.show()

In [ ]:
# GF = 0 from 6 h before exit, except a GF = 1 pulse of 2 h at `starts`.
starts = np.arange(0.0, 16.0, 2.0)
Y = np.tile(start, (1, starts.size))
extra = np.zeros(starts.size, dtype=int)
prev = Y[IB].copy()
for k in range(int(96.0 / h)):
    tk = -6.0 + k * h
    Y = model.integrate(Y, model.params({"GF": np.where((tk >= starts) & (tk < starts + 2.0), 1.0, 0.0)}), h, h)
    extra += (prev < 1) & (Y[IB] >= 1)
    prev = Y[IB].copy()
display(pd.DataFrame({"2 h GF pulse starts at (h after exit)": starts, "extra cycles": extra})
        .style.hide(axis="index"))

## 3. Coupling the model to the vertex model

Each face of the vertex model carries its own state vector
$\mathbf{y}_\alpha = (M_d, E2F, M_e, M_a, M_b, Cdc20)_\alpha$. The `SBMLCellCycle`
process advances all of them together, once per mechanics step.

**Mesh → model.** In this demonstration every cell sees the same growth factor,
GF = 1, so every cell cycles.
- The process can drive any SBML constant per cell from a `face_df` column, or from
  a ratio of two columns: `input_columns`, `input_functions`, `input_args`.
- Section 2c explains why such a coupling would lag by one to two division rounds.

**Model → mesh: phases decide growth and division.** Every step each cell's phase
is read from its own state by ordered rules (`phases`, first match wins, else G1):

| phase | rule | time after mitotic exit (h) | share of cycle | the cell |
|---|---|---|---|---|
| **S** | cyclin E > 1 | 4.4–13.1 | ~37% | holds its size |
| **M** | cyclin B > 1, cyclin E ≤ 1, cyclin A < 2 (cyclin A being destroyed) | 17.6–22.2 | ~19% | divides on entry, then holds |
| **G2** | cyclin B > 1, cyclin E ≤ 1 (cyclin A ≥ 2) | 13.1–17.6 | ~19% | grows |
| **G1** | otherwise | 22.2 → 4.4, across exit | ~25% | grows |

- **Growth.** Cells in G1 or G2 get the `grow_cells` behavior every step.
  - Their `prefered_area` grows at `growth_rate`, chosen so it doubles over the
    G1 + G2 time of one cycle.
  - Their `prefered_perimeter` grows as its square root, so the target shape is
    kept.
- **Division.** A cell that has just entered M gets the `split_cells` behavior.
  - It divides at once, whatever its size.
  - Both halves restart from target area 1 and the matching perimeter.
- **Initial sizes.** The initial cells start at the size their phase implies. A
  cell seeded late in G2 has already grown most of the way to 2.

**Inheritance.**
- Every step, the process writes each species to a `face_df` column `cc_<species>`,
  plus `cc_phase` (the phase) and `cc_last_division` (the time of the cell's
  latest division).
- The mother keeps her `unique_id`. `split_cells` stamps the daughter's
  `mother_uid`, and the process starts the daughter from her mother's state.
- The same columns are what the run's snapshots record.

**Heterogeneity.**
- The initial cells get random phases along the limit cycle.
- Each cell gets its own lognormal factor (CV 0.1) on $v_{sa}$, $v_{se}$ and
  $V_{de}$, the rates the period is most sensitive to. Sister cells therefore drift
  apart.

**Time.** `time_scale` hours of model time pass per unit of simulation time.

**A free border.**
- The tissue uses `model_factory`. `model_factory_bound` would zero the force on
  every `boundary` vertex and hold the border fixed, so here the sheet spreads as
  it grows.

In [ ]:
display(Markdown(
    f"**Configured:** GF = {sim.GROWTH_FACTOR}; `time_scale` = {sim.TIME_SCALE:.3f} "
    f"(one cycle ≈ {sim.CYCLE_TIME:g} sim time units); growth in {sim.GROWTH_PHASES} at "
    f"`growth_rate` = {sim.GROWTH_RATE:.4f}, division on entering {sim.DIVISION_PHASE}, "
    f"birth area {sim.BIRTH_AREA}; mechanics: prefered_perimeter {sim.PREF_PERIMETER}, "
    f"perimeter_elasticity {sim.PERIMETER_ELASTICITY}, viscosity {sim.VISCOSITY}, dt {sim.DT}; "
    f"run to {sim.TARGET_CELLS} cells, snapshots every {sim.FRAME_DT} time units; "
    f"parameter noise {sim.PARAMETER_NOISE}."
))

### The composite

The bigraph has two processes.
- `Tyssue` is the `EulerSolver`, which integrates the vertex model and owns the mesh.
- `Cell Cycle` is `SBMLCellCycle`. It reads `Tissue State` and pushes `grow_cells`,
  `split_cells` and `apply_gradient` behaviors into `Behaviors`.
- The solver executes those behaviors. The EulerSolver itself is unmodified.

In [ ]:
sys.path.insert(0, str(sim.REPO / "Experiments"))
from bigraph_style import plot_experiment_bigraph
display(plot_experiment_bigraph(
    sim.build_spec(sim.ensure_dataset()), core=core,
    filename="sbml_cell_cycle_bigraph", out_dir=sim.OUT_DIR / "bigraphs"))

## 4. The growing tissue

The run starts from the 206-cell flat square with a free border and GF = 1 in every
cell. It grows to 1200 cells (t ≈ 50, about two and a half cell cycles).

**Divisions start at once.** The initial phases are random, so some cell enters M
in every moment from the start. Each cell then divides once per cycle, about every
20 time units.

**Why the sheet stays clean.** A freely growing vertex sheet eventually folds:
faces pass through each other, because the vertex model has no term that resists
self-intersection.
- How large the sheet gets first is set by how fast the mechanics relax compared
  with growth: area elasticity ÷ viscosity (see `SWEEPS.md`).
  - With the earlier grow-then-split coupling, the first fold came at ~400 cells
    at viscosity 0.2 and at ~1500–1600 cells at the 0.02 used here.
- So the run stops at 1200 cells.
- Every 0.25-unit frame was checked with tyssue's exact `find_intersections`
  during the run. The last cell of this section checks the final mesh again.

In [ ]:
path = sim.OUT_DIR / sim.SCENARIO / "history.hf5"
with pd.HDFStore(str(path), "r") as store:
    data = {k.strip("/"): store.select(k) for k in store.keys()}
summary = pd.read_csv(sim.OUT_DIR / sim.SCENARIO / "summary.csv")
TIMES = np.sort(data["face"]["time"].unique())
last = summary.iloc[-1]
print(f"t = {last.time:.1f}: cells {int(summary.n_cells.iloc[0])} -> {int(last.n_cells)}, "
      f"{int(last.n_growing)} growing (G1 or G2), mean area {last.mean_area:.3f}")

In [ ]:
from scipy.spatial import ConvexHull
# Grouped straight from the archive: summary.csv times round-trip through text and
# no longer match the HDF5 times exactly. Both are sorted by time, so they align.
hull = np.array([ConvexHull(g[["x", "y"]].to_numpy()).volume
                 for _, g in data["vert"].groupby("time")])
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].plot(summary.time, summary.n_cells, color="k")
axes[0].set(xlabel="time", ylabel="cells", title="cell number")
# From the first update on: cc_phase is written by it, so at t = 0 every cell still
# reads phase 0 (G1) and would count as growing.
grow = summary[summary.time > TIMES[0]]
axes[1].plot(grow.time, grow.n_growing / grow.n_cells, color=COLORS["Mb"])
axes[1].set(xlabel="time", ylabel="fraction of cells", title="growing (G1 or G2)", ylim=(0, 1))
axes[2].plot(summary.time, hull, color="k", label="tissue footprint (convex hull)")
axes[2].plot(summary.time, summary.tissue_area, color="#7570b3", ls="--", label="sum of cell areas")
axes[2].set(xlabel="time", ylabel="area", title="tissue area")
axes[2].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "tissue_timeseries.png"); plt.show()

### Drawing helpers and fixed axes

The tissue grows, so every snapshot and every GIF frame uses the **same axes**:
the union of the vertex extents over all recorded frames, plus a margin. Each
cell's colour comes from its own model state.

**Phase colouring.** Each cell is coloured by the phase the process assigned it
(`cc_phase`, the rules in Section 3). G1 and G2 cells are growing; S and M cells
are holding their size.

**New cells.** A cell that divided within the last time unit, mother or daughter,
gets an outline: **magenta** on the phase colours, **orange** on the blue cyclin B
scale.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.animation import FuncAnimation, PillowWriter
from tyssue import Sheet, SheetGeometry

V = data["vert"]
pad = 0.05 * max(V["x"].max() - V["x"].min(), V["y"].max() - V["y"].min())
XLIM = (V["x"].min() - pad, V["x"].max() + pad)
YLIM = (V["y"].min() - pad, V["y"].max() + pad)
FIGSIZE = (7.0, 7.0 * (YLIM[1] - YLIM[0]) / (XLIM[1] - XLIM[0]))


def frame(t):
    t = TIMES[np.argmin(np.abs(TIMES - t))]
    dsets = {e: data[e][data[e]["time"] == t].drop(columns="time").set_index(e)
             for e in ("vert", "edge", "face")}
    sheet = Sheet("frame", dsets)
    SheetGeometry.update_all(sheet)
    sheet.frame_time = t
    return t, sheet


def polygons(sheet):
    # Face outlines, vertices sorted by angle around the face centroid. A frame
    # rebuilt from the snapshots has unordered edges, which tyssue's face_polygons
    # refuses; flat-sheet faces are star-shaped about their centroid, so the
    # angular sort recovers the outline.
    e = sheet.edge_df[["face", "sx", "sy"]].join(
        sheet.face_df[["x", "y"]].rename(columns={"x": "fx", "y": "fy"}), on="face")
    e["ang"] = np.arctan2(e["sy"] - e["fy"], e["sx"] - e["fx"])
    e = e.sort_values(["face", "ang"])
    return pd.Series({f: g[["sx", "sy"]].to_numpy() for f, g in e.groupby("face")})


NEW_EDGE = {"phase": "#e7298a",      # magenta on the phase colours
            "cyclin_B": "#e6550d"}   # orange, the complement of the blue cyclin B ramp
NEW_WINDOW = 1.0          # outline cells that divided within this long


def phase(fd):
    return np.array(sim.PHASE_NAMES, dtype=object)[fd["cc_phase"].to_numpy(dtype=int)]


CYCB_NORM = mcolors.LogNorm(1e-4, 3.0)
CYCB_CMAP = plt.get_cmap("Blues")    # single hue: light = low, dark = high


def draw(ax, sheet, mode):
    polys = polygons(sheet)
    fd = sheet.face_df.loc[polys.index]
    if mode == "phase":
        colors = [PHASE_COLORS[p] for p in phase(fd)]
    else:
        colors = CYCB_CMAP(CYCB_NORM(fd["cc_cyclin_B_Cdk1"].clip(lower=1e-4)))
    last = fd["cc_last_division"].to_numpy()
    new = (last > 0) & (sheet.frame_time - last < NEW_WINDOW)
    ax.add_collection(PolyCollection(list(polys.values), facecolors=colors,
                                     edgecolor="#555555", linewidths=0.3))
    if new.any():   # outline the cells that just divided, on top
        ax.add_collection(PolyCollection(list(polys.values[new]), facecolors="none",
                                         edgecolor=NEW_EDGE[mode], linewidths=1.1))
    ax.set_xlim(XLIM); ax.set_ylim(YLIM); ax.set_aspect("equal"); ax.axis("off")


def phase_legend(ax, **kw):
    handles = [Patch(facecolor=c, edgecolor="#555555", label=k) for k, c in PHASE_COLORS.items()]
    handles.append(Patch(facecolor="none", edgecolor=NEW_EDGE["phase"], linewidth=1.5,
                         label=f"divided < {NEW_WINDOW:g} ago"))
    ax.legend(handles=handles, frameon=False, fontsize=8, **kw)


def new_legend(ax, **kw):
    ax.legend(handles=[Patch(facecolor="none", edgecolor=NEW_EDGE["cyclin_B"], linewidth=1.5,
                             label=f"divided < {NEW_WINDOW:g} ago")], frameon=False, fontsize=8, **kw)
print(f"fixed axes: x {XLIM[0]:.1f}..{XLIM[1]:.1f}, y {YLIM[0]:.1f}..{YLIM[1]:.1f}")

### Snapshots

In [ ]:
# From t = 1: the cc_* columns are written by the first update, so t = 0 is blank.
snap_times = np.linspace(1.0, TIMES[-1], 4)
w, h = FIGSIZE[0] * 0.55, FIGSIZE[1] * 0.55
fig = plt.figure(figsize=(4 * w + 1.8, 2 * h))
# A fifth, narrow column holds the legend / colourbar, so all eight panels are
# the same size and share the same fixed axes.
gs = fig.add_gridspec(2, 5, width_ratios=[1, 1, 1, 1, 0.35], wspace=0.05, hspace=0.12)
for j, ts in enumerate(snap_times):
    t, sheet = frame(ts)
    a_top, a_bot = fig.add_subplot(gs[0, j]), fig.add_subplot(gs[1, j])
    draw(a_top, sheet, "phase")
    draw(a_bot, sheet, "cyclin_B")
    a_top.set_title(f"t = {t:.1f}   ({sheet.Nf} cells)", fontsize=9)
leg = fig.add_subplot(gs[0, 4]); leg.axis("off")
phase_legend(leg, loc="center left")
cax = fig.add_subplot(gs[1, 4])
cax.set_position([cax.get_position().x0, cax.get_position().y0 + 0.05,
                  0.012, cax.get_position().height - 0.1])
fig.colorbar(plt.cm.ScalarMappable(norm=CYCB_NORM, cmap=CYCB_CMAP), cax=cax,
             label="cyclin B (µM, per cell)")
fig.savefig(FIG_DIR / "snapshots_growth.png", bbox_inches="tight"); plt.show()

### GIFs

Two animations of the same run, both on the fixed axes, with about 160 frames:
- `growth_phase.gif`: each cell coloured by its cell-cycle phase;
- `growth_cyclinB.gif`: each cell coloured by its cyclin B.

In both, cells that divided within the last time unit are outlined: magenta in the
phase GIF, orange (the complement of the blue cyclin B scale) in the cyclin B GIF.

Both are written to `outputs/figures/`.

In [ ]:
from IPython.display import Image

N_GIF_FRAMES, GIF_FPS, GIF_DPI = 160, 12, 90
gif_times = TIMES[np.unique(np.linspace(0, len(TIMES) - 1, N_GIF_FRAMES).round().astype(int))]
gif_times = gif_times[gif_times >= 0.5]      # the cc_* columns are blank before the first update
frames_cache = {}


def cached_frame(t):
    if t not in frames_cache:
        frames_cache[t] = frame(t)[1]
    return frames_cache[t]


def make_gif(mode, filename):
    fig, ax = plt.subplots(figsize=(FIGSIZE[0] + 1.6, FIGSIZE[1]))
    fig.subplots_adjust(left=0.02, right=0.80, bottom=0.02, top=0.93)
    if mode == "phase":
        phase_legend(ax, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    else:
        fig.colorbar(plt.cm.ScalarMappable(norm=CYCB_NORM, cmap=CYCB_CMAP), ax=ax,
                     shrink=0.6, label="cyclin B (µM)")
        new_legend(ax, loc="upper left", bbox_to_anchor=(0.0, 1.0))

    def update(i):
        for coll in list(ax.collections):
            coll.remove()
        sheet = cached_frame(gif_times[i])
        draw(ax, sheet, mode)
        ax.set_title(f"t = {gif_times[i]:5.1f}    {sheet.Nf} cells", fontsize=11)
        return ax.collections

    anim = FuncAnimation(fig, update, frames=len(gif_times), blit=False)
    anim.save(FIG_DIR / filename, writer=PillowWriter(fps=GIF_FPS), dpi=GIF_DPI)
    plt.close(fig)
    return FIG_DIR / filename


gif_phase = make_gif("phase", "growth_phase.gif")
gif_cycb = make_gif("cyclin_B", "growth_cyclinB.gif")
print(gif_phase, gif_cycb, sep="\n")
display(Image(filename=str(gif_phase)))

In [ ]:
display(Image(filename=str(gif_cycb)))

### A lineage: the model state is inherited across division

`split_cells` records each daughter's mother in `mother_uid`, and the process
starts the daughter from her mother's state. So the lineage can be read straight
from the snapshots.

Below, cyclin B is traced through the generations of one lineage.

In [ ]:
F = data["face"].sort_values("time")
cols = [f"cc_{s}" for s in model.state_ids]
first = F.groupby("unique_id").head(1).set_index("unique_id")
t0 = TIMES[0]

born = first[first["time"] > t0]
parent = {int(u): int(m) for u, m in born["mother_uid"].items() if m >= 0}
children = {}
for c, p in parent.items():
    children.setdefault(p, []).append(c)
print(f"{len(parent)} of {int((first['time'] > t0).sum())} daughters matched to a mother")

def size(u):
    return 1 + sum(size(c) for c in children.get(u, []))
roots = [u for u in first.index[first["time"] == t0] if u in children]
root = max(roots, key=size)

fig, ax = plt.subplots(figsize=(9, 3.4))
gen_colors = ["#e7298a", "#7570b3", "#1b9e77", "#d95f02"]
seen = set()
def trace(u, g):
    tr = F[F["unique_id"] == u]
    ax.plot(tr["time"], tr["cc_cyclin_B_Cdk1"], color=gen_colors[g % 4], lw=1.2,
            label=f"generation {g}" if g not in seen else None)
    seen.add(g)
    if g > 0:
        ax.plot(tr["time"].iloc[0], tr["cc_cyclin_B_Cdk1"].iloc[0], "o", ms=3, color=gen_colors[g % 4])
    for c in children.get(u, []):
        trace(c, g + 1)
trace(root, 0)
ax.set(yscale="log", ylim=(1e-5, 5), xlabel="time", ylabel="cyclin B (µM)",
       title=f"lineage of cell {root}: daughters (dots) start from the mother's state")
ax.legend(frameon=False, fontsize=8, ncol=4)
fig.tight_layout(); fig.savefig(FIG_DIR / "lineage.png"); plt.show()

### Is the final mesh clean?

tyssue's exact detector looks for three things:
- faces whose boundary crosses itself;
- vertices inside a face they don't belong to;
- overlapping face pairs (`strict=True`).

In [ ]:
t_end, sheet_end = frame(TIMES[-1])
report = sheet_end.find_intersections(strict=True)
print(f"t = {t_end:.1f}: {sheet_end.Nf} cells, {len(report.folded)} folded, "
      f"{len(report.contained)} invading vertices, {len(report.overlaps)} overlapping pairs "
      f"-> {'clean' if report.clean else 'NOT clean'}")

## Limitations

- **No size checkpoint.** The cycle runs on its own clock. Growth is prescribed by
  phase (the target area doubles across G1 + G2), and a cell divides on entering M
  whatever size it has actually reached.
- **No Cdk inhibitors (p21, p27) or pRB.** There is no DNA-damage checkpoint and no
  molecular contact-inhibition pathway.
- **Commitment is set just before the previous mitotic exit (Section 2c).** Any
  signal coupled to GF acts one to two division rounds late. Models with a G1
  restriction point (BIOMD0000000730, Conradie 2010 BIOMD0000000265) shorten that
  lag, and they are drop-in replacements through `model_source`.
- **Deterministic** apart from the random initial phases and per-cell parameter
  noise. Runs are reproducible: `seed_all()` seeds numpy and Python's `random`.
- **The vertex model has no term that resists self-intersection.** A freely growing
  sheet still folds eventually: at ~1500–1600 cells with these mechanics, and much
  sooner with slower relaxation. That is why the run stops at 1200 cells. Growing
  it indefinitely needs an anti-self-intersection term in the mechanics.